# Problem Set 4
## Problem 1: Expectation Maximization for Cuts

In [4]:
# Gibbs Sampler
import numpy as np

def gibbs(A, s, t, w, burnin, its):
    """Gibbs Sampling algorithm

    Arguments:
        A {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        burnin {integer} -- number of burn-in samples
        its {integer} -- number of iterations after burn-in

    Returns:
        A vector of marginals whose ith component is equal to the probability
        that i ∈ V is in the same part of the cut as s.
    """

    n = A.shape[0]
    steps = burnin + its

    # Start with a valid sample
    sample = np.zeros(n)
    sample[s] = 1
    samples = np.empty((its, n))

    # We'll add any samples after burnin to totals
    totals = np.zeros(n)

    vertices = np.arange(n)
    # Don't consider changing s or t
    vertices = vertices[vertices != s]
    vertices = vertices[vertices != t]


    neighbors = [[idx for idx, edge in enumerate(A[i]) if edge == 1] for i in range(n)]

    for step in range(0, steps):
        # Randomize the order every time
        np.random.shuffle(vertices)

        for i in vertices:
            prob0 = 0
            prob1 = 0

            # Probability of x_i increases by w[i][j] whenever i != j
            for j in neighbors[i]:
                if sample[j] == 1:
                    prob0 += w[i][j]
                else:
                    prob1 += w[i][j]

            # Normalize
            prob1 = prob1 / (prob0 + prob1)

            # Stochastically update sample[i]
            # based on the marginal of i
            # that we calculated
            sample[i] = int(np.random.rand() < prob1)

        if step >= burnin:
            totals += sample
            samples[step - burnin] = sample

    return totals / its, samples

In [5]:
# Error Function
def error_function(prev, curr):
    error = sum([abs(curr[key] - prev[key]) for key in curr])
    return error / len(curr)

In [54]:
from collections import defaultdict
from copy import deepcopy

def compute_ess_for_cuts(edges, theta, D, L):
    n = len(edges)

    M = defaultdict(lambda: 0)

    def P(edge, x_ij, m):
        # Sample at m
        o_m = D[m]
        i, j = edge

        Q = 0
        # Fully observable
        if L[i] == 0 and L[j] == 0:
            # We XOR here because x_ij == 1 is only counting
            # instances where o_m[i] != o_m[j]
            # and x_ij == 0 is only counting
            # instances where o_m[i] == o_m[j]
            Q = int((x_ij == 1) ^ (o_m[i] == o_m[j]))

        # Partially observable
        # One or both of i/j is latent
        elif L[i] == 0 and L[j] == 1:
            for x_j in range(2):
                Q += theta[(i, j), x_ij] * int((x_ij == 1) ^ (o_m[i] == x_j))
        elif L[i] == 1 and L[j] == 0:
            for x_i in range(2):
                Q += theta[(i, j), x_ij] * int((x_ij == 1) ^ (x_i == o_m[j]))
        elif L[i] == 1 and L[j] == 1:
            for x_i in range(2):
                for x_j in range(2):
                    Q += theta[(i, j), x_ij] * int((x_ij == 1) ^ (x_i == x_j))

        return Q

    # Collect probabilities from all instances
    for m in range(len(D)):

        # Run inference on 〈G, θ〉 using evidence o[m]
        for (i, j) in edges:
            for x_ij in range(2):
                M[(i, j), x_ij] += P((i, j), x_ij, m)
    return M

In [76]:
# Color EM
def colorem(A, L, samples):
    """Expectation Maximization for Cuts

    Arguments:
        A {n x n matrix} -- adjacency matrix of graph G
        L {n-dimensional vector} -- latent variables
        samples {n x m matrix} --
            A list of m samples from which we'll
            estimate the weights of the edges in G

    Returns:
        The vector of weights w corresponding to the MLE
        parameters from each edge from the EM algorithm
    """
    n = len(A)
    m = samples.shape[0]

    edges = []
    for i in range(n):
        for j in range(n):
            if A[i,j] == 1 and (j,i) not in edges:
                edges.append((i,j))

    # Initial parameters
    theta = defaultdict(lambda: 0.5)

    while True:
        theta_prev = deepcopy(theta)

        # E-step
        # Expected sufficient statistics
        M = compute_ess_for_cuts(edges, theta, samples, L)

        # M-step
        for (i, j) in edges:
            for x_ij in range(2):
                theta[(i, j), x_ij] = M[(i, j), x_ij] / (M[(i, j), 0] + M[(i, j), 1])

        if error_function(theta_prev, theta) < 1e-30:
            break
    return theta

In [85]:
from pprint import pprint
#              a, b, c, d
A = np.array([[0, 1, 1, 0],  # a
              [1, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # b
              dtype = np.float64)

#              a, b, c, d
w = np.array([[0, 2, 1, 0],  # a
              [2, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # d
              dtype = np.float64)

s = 0
t = 3
its = 2**12
burnin = 2**12

samples = gibbs(A, s, t, w, its, burnin)[1]
L = [0, 0, 0, 0]
weights = colorem(A, L, samples)

print("Approximate Weights:")
pprint(weights)

Approximate Weights:
defaultdict(<function colorem.<locals>.<lambda> at 0x7aacc462cf70>,
            {((0, 1), 0): 0.325439453125,
             ((0, 1), 1): 0.674560546875,
             ((0, 2), 0): 0.506591796875,
             ((0, 2), 1): 0.493408203125,
             ((1, 3), 0): 0.674560546875,
             ((1, 3), 1): 0.325439453125,
             ((2, 3), 0): 0.493408203125,
             ((2, 3), 1): 0.506591796875})


In [86]:
from pprint import pprint
import math
#              a, b, c
A = np.array([[0, 1, 0],
              [1, 0, 1],
              [0, 1, 0]],
              dtype = np.float64)

#              a, b, c, d
w = np.array([[0, math.exp(1), 0],
              [math.exp(1), 0, math.exp(1)],
              [0, math.exp(1), 0]],
              dtype = np.float64)

s = 0
t = 2
its = 1000
burnin = 1000

samples = gibbs(A, s, t, w, its, burnin)[1]
L = [0, 0, 0]
weights = colorem(A, L, samples)

print("Approximate Weights:")
pprint(weights)

Approximate Weights:
defaultdict(<function colorem.<locals>.<lambda> at 0x7aaca843a710>,
            {((0, 1), 0): 0.497,
             ((0, 1), 1): 0.503,
             ((1, 2), 0): 0.503,
             ((1, 2), 1): 0.497})


## Problem 2: EM for Bayesian Networks
### 1. Fit a Bayesian Network

In [1]:
import pandas as pd
import numpy as np

voting_data = pd.read_csv('house-votes-84.data', header=None)
voting_data = voting_data.iloc[:, :3]
voting_data[voting_data == '?'] = None
voting_data[voting_data == 'republican'] = 0
voting_data[voting_data == 'democrat'] = 1
voting_data[voting_data == 'n'] = 0
voting_data[voting_data == 'y'] = 1

print(voting_data[:20])

train = np.array(voting_data.iloc[:300, :3])
test = np.array(voting_data.iloc[300:, :3])

possible_DAGs = [
    { 0: [], 1: [0], 2: [0,1] },
    { 0: [], 1: [0,2], 2: [0] },
    { 0: [1], 1: [], 2: [0,1] },
    { 0: [1,2], 1: [], 2: [1] },
    { 0: [2], 1: [0,2], 2: [] },
    { 0: [1,2], 1: [2], 2: [] },
]


    0     1     2
0   0     0     1
1   0     0     1
2   1  None     1
3   1     0     1
4   1     1     1
5   1     0     1
6   1     0     1
7   0     0     1
8   0     0     1
9   1     1     1
10  0     0     1
11  0     0     1
12  1     0     1
13  1     1     1
14  0     0     1
15  0     0     1
16  1     1     0
17  1     1  None
18  0     0     1
19  1     1     1


In [18]:
from itertools import product

def generate_permutations(sample):
    indices = [i for i, v in enumerate(sample) if v is None]
    for perm in product([0,1], repeat=len(indices)):
        if perm != ():
            result = sample.copy()
            for idx, val in zip(indices, perm):
                result[idx] = val
            yield result

In [70]:
from collections import defaultdict
from copy import deepcopy

def compute_ess(G, theta, D):
    n = len(G)

    M = defaultdict(lambda: 0)

    def P(i, x_i, u_i, m):
        # Sample at m
        o_m = D[m]
        # Parents
        u = G[i]

        # Sum of probs over all possible permutations of o_m
        # when o_m[i] = x_i and o_m[u] = u_i
        if o_m[i] == None or None in o_m[u]:
            #   P(x_i, x_u | o_m, theta)
            # = Q(x_i, x_u)
            # = Sum of Q(x_?, x_i, x_u) over Val(x_?, x_i, x_u)
            sum_Q = 0

            missing_indices = np.where(o_m == None)[0]

            # Sum the probabilities that each None takes a specific value
            for instantiation in generate_permutations(o_m):
                if x_i == instantiation[i] and u_i == tuple(instantiation[u]):
                    prob = 1
                    for j in missing_indices:
                        x_j = instantiation[j]
                        v = G[j]
                        v_i = instantiation[v]
                        prob *= theta[j, x_j, tuple(v_i)]
                    sum_Q += prob

            return sum_Q

        # Fully observable case
        # 1_{ξ[m]〈C〉= c^1}
        return int(o_m[i] == x_i and tuple(o_m[u]) == u_i)

    # Collect probabilities from all instances
    for m in range(len(D)):

        # Run inference on 〈G, θ〉 using evidence o[m]
        for i in range(n):
            for x_i in range(2):
                for u_i in list(product([0,1], repeat=len(G[i]))):
                    M[i, x_i, u_i] += P(i, x_i, u_i, m)
    return M

In [144]:
def bayesian_em(G, samples):
    n = len(G)
    m = samples.shape[0]

    # Initial parameters
    theta = defaultdict(lambda: 0.5)

    while True:
        theta_prev = deepcopy(theta)

        # E-step
        # Expected sufficient statistics
        M = compute_ess(G, theta, samples)

        # M-step
        for i in range(n):
            for x_i in range(2):
                for u_i in list(product([0,1], repeat=len(G[i]))):
                    theta[i, x_i, u_i] = M[i, x_i, u_i] / (M[i, 0, u_i] + M[i, 1, u_i])

        if error_function(theta_prev, theta) < 1e-20:
            break
    return theta

In [132]:
def evaluate(G, theta, data):
    correct = 0
    wrong = 0

    for it, x in enumerate(data):
        label = x[0]
        features = x[1:]

        P_0 = 0
        P_1 = 0
        # Partially observable
        if None in features:
            # Add 1 here so the indices are correct after we add the first column back in
            missing_indices = np.where(features == None)[0] + 1

            for partial_instantiation in generate_permutations(features):
                instantiation_0 = np.append([0], partial_instantiation)
                instantiation_1 = np.append([1], partial_instantiation)
                prob0 = 1
                prob1 = 1

                for i in missing_indices:
                    x_i = instantiation_0[i]
                    u = G[i]
                    u_i = instantiation_0[u]
                    prob0 *= theta[i, x_i, tuple(u_i)]
                P_0 += prob0

                for i in missing_indices:
                    x_i = instantiation_1[i]
                    u = G[i]
                    u_i = instantiation_1[u]
                    prob1 *= theta[i, x_i, tuple(u_i)]
                P_1 += prob1
        # Fully observable
        else:
            P_0 = 1
            P_1 = 1

        u = G[0]
        u_i = x[u]
        P_0 *= theta[0, 0, tuple(u_i)]
        P_1 *= theta[0, 1, tuple(u_i)]

        predicted = int(P_0 < P_1)
        # if it > 116 and it < 120:
        #     print(P_0)
        #     print(P_1)
        #     print(tuple(G[0]))
        #     print(it)
        #     print(f"Pred: {'democrat' if predicted == 1 else 'republican'}")
        #     print(f"Actual: {'democrat' if label == 1 else 'republican'}")
        #     print()
        if predicted == label:
            correct += 1
        else:
            wrong += 1

    return correct / (correct + wrong)

In [69]:
x = pd.read_csv('house-votes-84.data', header=None)
x = x.iloc[:, :3][:300]
print(pd.DataFrame(train).value_counts([0], dropna=False))
pd.DataFrame(train).value_counts([0, 1, 2], dropna=False)

1    187
0    113
dtype: int64


0  1    2  
1  1.0  0.0    54
        1.0    45
0  0.0  1.0    43
1  0.0  0.0    36
0  0.0  0.0    35
1  0.0  1.0    28
0  0.0  NaN    13
1  1.0  NaN    10
0  1.0  1.0     9
        0.0     8
1  0.0  NaN     7
0  1.0  NaN     4
1  NaN  NaN     4
        1.0     2
0  NaN  NaN     1
1  NaN  0.0     1
dtype: int64

In [83]:
theta = defaultdict(lambda: 0.5)
compute_ess(possible_DAGs[0], theta, train)

defaultdict(<function __main__.compute_ess.<locals>.<lambda>()>,
            {(0, 0, ()): 113,
             (0, 1, ()): 187,
             (1, 0, (0,)): 91.5,
             (1, 0, (1,)): 74.5,
             (1, 1, (0,)): 21.5,
             (1, 1, (1,)): 112.5,
             (2, 0, (0, 0)): 41.75,
             (2, 0, (0, 1)): 10.25,
             (2, 0, (1, 0)): 41.0,
             (2, 0, (1, 1)): 60.5,
             (2, 1, (0, 0)): 49.75,
             (2, 1, (0, 1)): 11.25,
             (2, 1, (1, 0)): 33.5,
             (2, 1, (1, 1)): 52.0})

In [137]:
pd.DataFrame(test).value_counts([0], dropna=False)

1    80
0    55
dtype: int64

### 2. Do different runs of the EM algorithm produce different models?
The EM algorithm implementation I used is deterministic. Given the same data and DAG, the same model will be produced. Another implementation of the EM algorithm might randomize the initial theta parameters. Assuming the function to be maximized has more than one local maximum, each run may converge on different parameters for theta. A random EM algorithm may still converge on the same parameters across multiple runs in the case that there is a single local maximum or the initial theta parameters share the same local maximum.

In [ ]:
from pprint import pprint

for dag in possible_DAGs:
    theta = bayesian_em(dag, train)
    pprint(theta)
    print()


defaultdict(<function bayesian_em.<locals>.<lambda> at 0x7394aa468af0>,
            {(0, 0, ()): 0.37666666666666665,
             (0, 1, ()): 0.6233333333333333,
             (1, 0, (0,)): 0.8125,
             (1, 0, (1,)): 0.3944444444444445,
             (1, 1, (0,)): 0.1875,
             (1, 1, (1,)): 0.6055555555555555,
             (2, 0, (0, 0)): 0.4487179487179488,
             (2, 0, (0, 1)): 0.47058823529411764,
             (2, 0, (1, 0)): 0.5583397255603852,
             (2, 0, (1, 1)): 0.541632225712239,
             (2, 1, (0, 0)): 0.5512820512820512,
             (2, 1, (0, 1)): 0.5294117647058824,
             (2, 1, (1, 0)): 0.44166027443961475,
             (2, 1, (1, 1)): 0.458367774287761})
Accuracy: 0.5925925925925926

defaultdict(<function bayesian_em.<locals>.<lambda> at 0x7394aa46b0a0>,
            {(0, 0, ()): 0.37666666666666665,
             (0, 1, ()): 0.6233333333333333,
             (1, 0, (0, 0)): 0.8064784053156145,
             (1, 0, (0, 1)): 0.8174793

### 3. Evaluation
The predictions are dependent on the model that was fit. The models where party is dependent on vote 1 (models 3, 4, and 6) perform significantly better than other models.

In models 1 and 2, party is independent of votes 1 and 2. Therefore, these models predict the most frequent party from the training data, regardless of any votes. In this case, the training data has more Democrats. The testing data has 80 Democrats and 55 Republicans, therefore predicting Democrat every time results in the accuracies of 0.5926 shown below.

As seen in the difference between models 3 and 4, vote 2 seems to improve predictions when party is already dependent on vote 1. However, in model 5, predictions are negatively impacted when party is dependent on vote 2 alone. In this case, the model would have performed better if party was independent.

In [160]:
for i, dag in enumerate(possible_DAGs):
    theta = bayesian_em(dag, train)
    print(f"Model {i + 1}")
    print('  |  '.join(f"{dag[vertex]} -> {vertex}" for vertex in dag))
    print(f"Accuracy: {evaluate(dag, theta, test)}")
    print()

# theta = bayesian_em(possible_DAGs[3], train)
# pprint(theta)
# print(f"Accuracy: {evaluate(possible_DAGs[3], theta, test)}")

Model 1
[] -> 0  |  [0] -> 1  |  [0, 1] -> 2
Accuracy: 0.5925925925925926

Model 2
[] -> 0  |  [0, 2] -> 1  |  [0] -> 2
Accuracy: 0.5925925925925926

Model 3
[1] -> 0  |  [] -> 1  |  [0, 1] -> 2
Accuracy: 0.6814814814814815

Model 4
[1, 2] -> 0  |  [] -> 1  |  [1] -> 2
Accuracy: 0.6592592592592592

Model 5
[2] -> 0  |  [0, 2] -> 1  |  [] -> 2
Accuracy: 0.5555555555555556

Model 6
[1, 2] -> 0  |  [2] -> 1  |  [] -> 2
Accuracy: 0.6592592592592592

